In [ ]:
#cell 1
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Bidirectional, LSTM, Input, Add, Softmax, multiply, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
tf.random.set_seed(42)

TARGET_COL = "CPU usage [%]"

# ===============================
# FEDERATED CONFIG
# ===============================

FL_CONFIG = {
    "num_rounds": 10,
    "local_epochs": 6,
    "batch_size": 32,
    "sequence_length": 32,
    "prediction_steps": 1,
    "learning_rate": 0.001
}

# ===============================
# MODEL UNITS CONFIG
# ===============================

MODEL_CONFIG = {
    "lstm_units": 40,
    "gru_units": 40,
    "bilstm_units": 40,
    "bigru_units": 40
}


#Creating groups

TOTAL_FILES = 519
GROUP_SIZE = 5

file_groups = []

for start in range(1, TOTAL_FILES+1, GROUP_SIZE):

    group = list(range(start, start + GROUP_SIZE))

    if group[-1] <= TOTAL_FILES:
        file_groups.append(group)


print("Total runs:", len(file_groups))
print("Example groups:", file_groups[:20])

In [ ]:
for CLIENT_FILE_NUMBERS in file_groups:

    print("\n=======================================")
    print("Running experiment for files:", CLIENT_FILE_NUMBERS)
    print("=======================================\n")



    #cell 2
    
    DATA_DIR = "/kaggle/input/datasets/darshanhegde17/materna-t13/Materna-Trace-1"
    
    
    all_csv_files = sorted(
        glob.glob(os.path.join(DATA_DIR, "*.csv")),
        key=lambda x: int(os.path.basename(x).replace(".csv",""))
    )
    
    csv_files = [all_csv_files[i-1] for i in CLIENT_FILE_NUMBERS]
    
    print("Number of clients:", len(csv_files))

    
    # CELL 3 — Load RAW (NO SCALING)
    
    def load_client_data(path):
        df = pd.read_csv(path, sep=";", low_memory=False)
        df[TARGET_COL] = df[TARGET_COL].astype(str).str.replace(",", ".").astype(float)
        df[CORES_COL] = df[CORES_COL].astype(str).str.replace(",", ".").astype(float)
        df = df.ffill().bfill()
        y_raw = df[[TARGET_COL, CORES_COL]].values
        return y_raw

    
    
    def make_sequences(y, L, H):    
        Xs, ys = [], []
        for i in range(len(y)-L-H):
            Xs.append(y[i:i+L])
            ys.append(y[i+L:i+L+H, 0].flatten()) 
            
        return np.array(Xs), np.array(ys)

    
    
    # CELL 4 — Per-client split
    
    clients_data = []
    
    for path in csv_files:
    
        # -----------------------------
        # LOAD RAW DATA
        # -----------------------------
        y_raw = load_client_data(path)
    
        n_total = len(y_raw)
    
        # -----------------------------
        # SPLIT FIRST (RAW DOMAIN)
        # -----------------------------
        train_end = int(0.60 * n_total)
        val_end   = int(0.80 * n_total)
    
        y_train_raw = y_raw[:train_end]
        y_val_raw   = y_raw[train_end:val_end]
        y_test_raw  = y_raw[val_end:]
    
        # -----------------------------
        # SCALE (FIT ONLY ON TRAIN)
        # -----------------------------
        scaler = MinMaxScaler()
        scaler.fit(y_train_raw)
    
        y_train = scaler.transform(y_train_raw)
        y_val   = scaler.transform(y_val_raw)
        y_test  = scaler.transform(y_test_raw)
    
        # -----------------------------
        # CREATE SEQUENCES SEPARATELY
        # -----------------------------
        X_train, y_train_seq = make_sequences(
            y_train,
            FL_CONFIG["sequence_length"],
            FL_CONFIG["prediction_steps"]
        )
    
        X_val, y_val_seq = make_sequences(
            y_val,
            FL_CONFIG["sequence_length"],
            FL_CONFIG["prediction_steps"]
        )
    
        X_test, y_test_seq = make_sequences(
            y_test,
            FL_CONFIG["sequence_length"],
            FL_CONFIG["prediction_steps"]
        )
    
        clients_data.append({
            "X_train": X_train,
            "y_train": y_train_seq,
            "X_val": X_val,
            "y_val": y_val_seq,
            "X_test": X_test,
            "y_test": y_test_seq,
            "scaler": scaler,
            "num_samples": len(X_train)
        })
    
    print("Clients prepared:", len(clients_data))

    
    
    # CELL 5 — Model Architectures
    
    def create_lstm():
        u = MODEL_CONFIG["lstm_units"]
    
        model = Sequential([
            LSTM(u, activation="relu", return_sequences=True,
                 input_shape=(FL_CONFIG["sequence_length"],1)),
            LSTM(u, activation="relu", return_sequences=True),
            LSTM(u, activation="relu"),
            Dense(FL_CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(FL_CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    def create_gru():
        u = MODEL_CONFIG["gru_units"]
    
        model = Sequential([
            GRU(u, activation="relu", return_sequences=True,
                input_shape=(FL_CONFIG["sequence_length"],1)),
            GRU(u, activation="relu", return_sequences=True),
            GRU(u, activation="relu"),
            Dense(FL_CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(FL_CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    def create_bilstm():
        u = MODEL_CONFIG["bilstm_units"]
    
        model = Sequential([
            Bidirectional(LSTM(u, activation="relu", return_sequences=True),
                          input_shape=(FL_CONFIG["sequence_length"],1)),
            Bidirectional(LSTM(u, activation="relu", return_sequences=True)),
            Bidirectional(LSTM(u, activation="relu")),
            Dense(FL_CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(FL_CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    def create_bigru():
        u = MODEL_CONFIG["bigru_units"]
    
        model = Sequential([
            Bidirectional(GRU(u, activation="relu", return_sequences=True),
                          input_shape=(FL_CONFIG["sequence_length"],1)),
            Bidirectional(GRU(u, activation="relu", return_sequences=True)),
            Bidirectional(GRU(u, activation="relu")),
            Dense(FL_CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(FL_CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    all_models = {
        "LSTM": create_lstm,
        "GRU": create_gru,
        "BiLSTM": create_bilstm,
        "BiGRU": create_bigru
    }
    

    
    
    # CELL 6 — FedAvg
    
    def fedavg(weights, sizes):
    
        total = sum(sizes)
    
        return [
            sum(w[i]*(s/total) for w,s in zip(weights, sizes))
            for i in range(len(weights[0]))
        ]
    
        
    
    # CELL 7 — Federated Training
    
    trained_models = {}
    
    for name, model_fn in all_models.items():
    
        print("\n============================")
        print("Training Federated", name)
        print("============================")
    
        global_model = model_fn()
    
        for rnd in range(FL_CONFIG["num_rounds"]):
    
            print("Round", rnd+1)
    
            global_w = global_model.get_weights()
    
            local_ws = []
            sizes = []
    
            for client in clients_data:
    
                model = model_fn()
                model.set_weights(global_w)
    
                model.fit(
                    client["X_train"],
                    client["y_train"],
                    epochs=FL_CONFIG["local_epochs"],
                    batch_size=FL_CONFIG["batch_size"],
                    shuffle=False,
                    verbose=0
                )
    
                local_ws.append(model.get_weights())
                sizes.append(client["num_samples"])
    
            global_model.set_weights(fedavg(local_ws, sizes))
    
        trained_models[name] = global_model
    
    
    
    
    # CELL 8 — Federated Evaluation
    
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    
    def rmse(y, yhat):
        return np.sqrt(mean_squared_error(y, yhat))
    
    def rmsle(y, yhat):
        return np.sqrt(np.mean((np.log1p(y) - np.log1p(yhat))**2))
    
    
    def evaluate_federated(model):
    
        all_true_scaled = []
        all_pred_scaled = []
    
        all_true_real = []
        all_pred_real = []
    
        predictions_per_client = []
    
        # --------------------------------------
        # Evaluate each client
        # --------------------------------------
        for client in clients_data:
    
            pred_scaled = model.predict(client["X_test"], verbose=0)
            true_scaled = client["y_test"]
    
            scaler = client["scaler"]
    
            true_real = scaler.inverse_transform(true_scaled)
            pred_real = scaler.inverse_transform(pred_scaled)
    
            true_real = np.clip(true_real, 0, None)
            pred_real = np.clip(pred_real, 0, None)
    
            # store predictions
            predictions_per_client.append({
                "true_scaled": true_scaled,
                "pred_scaled": pred_scaled,
                "true_real": true_real,
                "pred_real": pred_real
            })
    
            all_true_scaled.append(true_scaled)
            all_pred_scaled.append(pred_scaled)
    
            all_true_real.append(true_real)
            all_pred_real.append(pred_real)
    
        # --------------------------------------
        # Concatenate all clients
        # --------------------------------------
        y_true_scaled = np.concatenate(all_true_scaled)
        y_pred_scaled = np.concatenate(all_pred_scaled)
    
        y_true_real = np.concatenate(all_true_real)
        y_pred_real = np.concatenate(all_pred_real)
    
        # --------------------------------------
        # Metrics
        # --------------------------------------
        results = {
            "scaled": {
                "MAE": mean_absolute_error(y_true_scaled, y_pred_scaled),
                "MSE": mean_squared_error(y_true_scaled, y_pred_scaled),
                "RMSE": rmse(y_true_scaled, y_pred_scaled),
                "RMSLE": rmsle(
                    np.clip(y_true_scaled,1e-8,None),
                    np.clip(y_pred_scaled,1e-8,None)
                )
            },
            "real": {
                "MAE": mean_absolute_error(y_true_real, y_pred_real),
                "MSE": mean_squared_error(y_true_real, y_pred_real),
                "RMSE": rmse(y_true_real, y_pred_real),
                "RMSLE": rmsle(y_true_real, y_pred_real)
            }
        }
    
        # --------------------------------------
        # Print results
        # --------------------------------------
        print("\n==============================")
        print("Federated Results")
        print("==============================")
    
        print("MAE_scaled   :", round(results["scaled"]["MAE"],6))
        print("MSE_scaled   :", round(results["scaled"]["MSE"],6))
        print("RMSE_scaled  :", round(results["scaled"]["RMSE"],6))
        print("RMSLE_scaled :", round(results["scaled"]["RMSLE"],6))
    
        print()
    
        print("MAE_real     :", round(results["real"]["MAE"],6))
        print("MSE_real     :", round(results["real"]["MSE"],6))
        print("RMSE_real    :", round(results["real"]["RMSE"],6))
        print("RMSLE_real   :", round(results["real"]["RMSLE"],6))
    
        return results, predictions_per_client
    
    
    all_results = {}
    all_predictions = {}
    
    for name, model in trained_models.items():
    
        print("\n------------------------------")
        print("Evaluating Federated", name)
        print("--------------------------------")
    
        results, preds = evaluate_federated(model)
    
        all_results[name] = results
        all_predictions[name] = preds
     
    
    
    # SAVING RESULTS
    
    import json
    
    base_folder = "Fed_Materna_results"
    os.makedirs(base_folder, exist_ok=True)
    
    dataset_name = f"{min(CLIENT_FILE_NUMBERS)}-{max(CLIENT_FILE_NUMBERS)}_seq{FL_CONFIG['sequence_length']}_federated"
    
    run_folder = os.path.join(base_folder, dataset_name)
    os.makedirs(run_folder, exist_ok=True)
    
    
    # ===============================
    # SAVE CONFIG
    # ===============================
    with open(os.path.join(run_folder, "config.json"), "w") as f:
        json.dump(FL_CONFIG, f, indent=4)
    
    
    # ===============================
    # SAVE METRICS
    # ===============================
    metrics_file = os.path.join(run_folder, "federated_metrics.json")
    
    with open(metrics_file, "w") as f:
        json.dump(all_results, f, indent=4)
    
    
    # ===============================
    # SAVE PREDICTIONS
    # ===============================
    
    rows = []
    
    for model_name in all_predictions.keys():
    
        for client_id, client_pred in enumerate(all_predictions[model_name]):
    
            true_scaled = client_pred["true_scaled"].flatten()
            pred_scaled = client_pred["pred_scaled"].flatten()
    
            true_real = client_pred["true_real"].flatten()
            pred_real = client_pred["pred_real"].flatten()
    
            for i in range(len(true_scaled)):
    
                rows.append({
                    "model": model_name,
                    "client": client_id,
                    "true_scaled": true_scaled[i],
                    "pred_scaled": pred_scaled[i],
                    "true_real": true_real[i],
                    "pred_real": pred_real[i]
                })
    
    
    df = pd.DataFrame(rows)
    
    pred_file = os.path.join(run_folder, "federated_predictions.csv")
    df.to_csv(pred_file, index=False)
    
    
    print("\n================================")
    print("Federated results saved in:", run_folder)
    print("================================")

In [ ]:
import shutil

# Replace this path with your actual folder path in the Output section
folder_path = '/kaggle/working/Materna_results'

# Creating a zip archive of the folder
shutil.make_archive(folder_path, 'zip', folder_path)

# The .zip file saved at /kaggle/working/ and you can download it from there